# Week 11: Express APIs & Middleware

This notebook introduces building HTTP APIs with **Express** and the key idea that powers it: **middleware**.

**Goal:** understand routing + request/response handling, then build reusable middleware for logging, auth, validation, and error handling.

> If you get `Cannot find module 'express'`, run `npm i express` in the repo root.

## Table of Contents

1. What is Express?
2. Minimal API server
3. Routing essentials (params, query, body)
4. Middleware: the pipeline model
5. Built-in middleware (`express.json()`, `express.urlencoded()`, static)
6. Custom middleware: logging + request id
7. Custom middleware: auth (Bearer token)
8. Routers: modular APIs
9. Error handling: 404 + centralized error middleware
10. Exercises

## 1) What is Express?

- A minimal web framework for Node.js.
- You define **routes** (`GET /users`, `POST /items`) and attach **middleware** that runs before/after them.

Express runs on top of Node’s `http` module but provides:
- A routing layer
- Convenience helpers on `req` and `res`
- Middleware composition

Docs: [Express](https://expressjs.com/)

## 2) Minimal API server

**Run it:** put the next cell into a file like `week11/lecture/express-server.js`, then:

```bash
node week11/lecture/express-server.js
```

Open `http://localhost:3000/health`.

In [ ]:
const express = require('express');

const app = express();
const PORT = 3000;

app.get('/health', (req, res) => {
  res.json({ ok: true, uptime: process.uptime() });
});

app.listen(PORT, () => {
  console.log(`API listening on http://localhost:${PORT}`);
});

## 3) Routing essentials

### Params
- `GET /users/:id` → `req.params.id`

### Query string
- `GET /search?q=js&page=2` → `req.query.q`, `req.query.page`

### Body
- For JSON request bodies you must use `app.use(express.json())`.

### Response helpers
- `res.status(201).json(data)`
- `res.send('text')`
- `res.set('Header-Name', 'value')`

### Simple CRUD example (Express)
See [Упражнение 7 (2024/2025)](https://github.com/FMIjs/advanced-javascript-2024-2025/blob/master/week7/README.md#%D1%83%D0%BF%D1%80%D0%B0%D0%B6%D0%BD%D0%B5%D0%BD%D0%B8%D0%B5-7) for a clean “CRUD + nested resource” API design:
- **Events**: `POST /event`, `GET /event/:id`, `DELETE /event/:id`
- **Bookings (nested under an event)**: `POST /event/:id/booking`, `GET /event/:id/booking`, `GET /event/:id/booking/:bookingId`, `DELETE /event/:id/booking/:bookingId`

Example of using route params to model relationships like **event → bookings**.

In [ ]:
const express = require('express');

const app = express();
app.use(express.json());

const users = new Map([
  [1, { id: 1, name: 'Ada' }],
  [2, { id: 2, name: 'Grace' }],
]);
let nextId = 3;

// GET /users
app.get('/users', (req, res) => {
  res.json([...users.values()]);
});

// GET /users/:id
app.get('/users/:id', (req, res) => {
  const id = Number(req.params.id);
  const user = users.get(id);
  if (!user) return res.status(404).json({ error: 'User not found' });
  res.json(user);
});

// POST /users  { "name": "..." }
app.post('/users', (req, res) => {
  const name = String(req.body?.name ?? '').trim();
  if (!name) return res.status(400).json({ error: 'name is required' });

  const user = { id: nextId++, name };
  users.set(user.id, user);
  res.status(201).json(user);
});

app.listen(3000, () => console.log('http://localhost:3000'));

## 4) Middleware: the pipeline model

**Middleware is just a function** that runs in order.

Signature:
```js
(req, res, next) => {
  // do something
  next(); // pass control to the next middleware/route
}
```

Key ideas:
- Order matters: `app.use(...)` runs top-to-bottom.
- Middleware can **end** the request (by sending a response) or **pass** it forward (`next()`).
- Middleware can be global (`app.use`) or attached to a route (`app.get('/x', mw, handler)`).

## 5) Built-in middleware

- `express.json()` parses JSON bodies into `req.body`
- `express.urlencoded({ extended: true })` parses HTML form bodies
- `express.static('public')` serves static files

Example:
```js
app.use(express.json());
app.use(express.urlencoded({ extended: true }));
app.use('/static', express.static('public'));
```

## 6) Custom middleware: logging + request id

A simple production-friendly pattern:
- Assign `req.id`
- Measure request duration
- Log method, path, status, and duration

### More middleware examples (custom body + form parsers)
See the lecture example in [week8/lecture/index.js (2024/2025)](https://github.com/FMIjs/advanced-javascript-2024-2025/blob/master/week8/lecture/index.js).

It demonstrates a few classic middleware ideas:
- **Stream parsing helper**: `formParser(req)` reads the request stream, collects chunks, then parses `key=value&...` into an object.
- **Conditional middleware**: `bodyParser(req, res, next)` checks `Content-Type` and only parses when it’s `application/x-www-form-urlencoded`, then sets `req.body`.
- **Error propagation**: parse errors are forwarded with `next(err)` so a centralized error handler can handle them.

In [ ]:
const express = require('express');

const app = express();

let requestSeq = 0;

app.use((req, res, next) => {
  const id = String(++requestSeq);
  req.id = id; // ad-hoc property; in TS you'd extend the Request type

  const start = Date.now();
  res.on('finish', () => {
    const ms = Date.now() - start;
    console.log(`[${id}] ${req.method} ${req.originalUrl} -> ${res.statusCode} (${ms}ms)`);
  });

  next();
});

app.get('/hello', (req, res) => {
  res.json({ message: 'hello', requestId: req.id });
});

app.listen(3000, () => console.log('http://localhost:3000'));

## 7) Custom middleware: auth (Bearer token)

A common approach is:
- Read the `Authorization: Bearer <token>` header
- Validate token
- Attach `req.user` and call `next()`
- If invalid/missing, return `401`/`403`

### Auth example
See the exercise: [week13/exercise/auth (2024/2025)](https://github.com/FMIjs/advanced-javascript-2024-2025/tree/master/week13/exercise/auth).

- **Auth endpoint(s)** (e.g. login) that issue some credential/token
- **Auth middleware** that validates the credential on each request
- **Protected routes** that only work when auth passes (otherwise `401`/`403`)

In [ ]:
const express = require('express');

const app = express();

function requireBearerToken(validToken) {
  return (req, res, next) => {
    const header = String(req.headers.authorization ?? '');
    const [scheme, token] = header.split(' ');

    if (scheme !== 'Bearer' || !token) {
      return res.status(401).json({ error: 'Missing Bearer token' });
    }

    if (token !== validToken) {
      return res.status(403).json({ error: 'Invalid token' });
    }

    req.user = { id: 'demo-user' };
    next();
  };
}

app.get('/public', (req, res) => res.json({ ok: true, scope: 'public' }));

app.get('/private', requireBearerToken('secret123'), (req, res) => {
  res.json({ ok: true, scope: 'private', user: req.user });
});

app.listen(3000, () => console.log('Try GET /public and GET /private'));

## 8) Routers: modular APIs

As your API grows, group endpoints into routers and mount them:

```js
const api = express.Router();
api.get('/users', ...);
app.use('/api', api);
```

Router-level middleware is especially useful (e.g. `api.use(requireAuth)` for everything under `/api`).

## 9) Error handling: 404 + centralized error middleware

Two best practices:
- Put a **404 handler** after all routes.
- Put a single **error-handling middleware** at the end.

Error middleware signature has **4 args**:
```js
(err, req, res, next) => { ... }
```

In [ ]:
const express = require('express');

const app = express();
const api = express.Router();

function asyncHandler(fn) {
  return (req, res, next) => Promise.resolve(fn(req, res, next)).catch(next);
}

api.get('/ping', (req, res) => res.json({ ok: true }));

api.get('/boom', asyncHandler(async (req, res) => {
  await new Promise(r => setTimeout(r, 10));
  throw new Error('Something went wrong');
}));

app.use('/api', api);

// 404 (must be AFTER routes)
app.use((req, res) => {
  res.status(404).json({ error: 'Not found', path: req.originalUrl });
});

// error handler (must be LAST)
app.use((err, req, res, next) => {
  console.error('Unhandled error:', err);
  res.status(500).json({ error: 'Internal Server Error' });
});

app.listen(3000, () => console.log('Try GET /api/ping, /api/boom, /missing'));

## 10) Exercises

1. **CRUD API**: implement `GET /items`, `GET /items/:id`, `POST /items`, `PUT /items/:id`, `DELETE /items/:id` using an in-memory array.
2. **Validation middleware**: create `validateBody(schema)` that checks required fields and responds with `400` on invalid input.
3. **Auth middleware**: implement `requireRole(role)` and add `req.user = { role: 'admin' | 'user' }` to simulate authorization.
4. **Error handling**: throw errors inside async route handlers and make sure they reach your error middleware.

**Tip:** You can use `curl` to test quickly instead of `Postman`:
```bash
curl -s http://localhost:3000/health
curl -s -X POST http://localhost:3000/users -H 'Content-Type: application/json' -d '{"name":"Linus"}'
```